# 第四部分：超级提示词的崩塌 (The Mega-Prompt Collapse)
### *Alex 的野心：一个 Prompt 统治所有规则*

Alex 决定把公司所有的“智慧”都塞进 Agent。他写了一个涵盖所有场景的超级指令（Mega-Prompt）。  

在这个笔记本中，我们将亲身体验：
1. **复杂性爆炸**：当系统指令包含几十条细碎规则时，代码的可读性如何降为零。
2. **注意力稀释**：观察 Agent 是否会因为指令太长而出现“幻觉”或遗漏规则。
3. **Token 浪费**：计算这种“全量注入”模式下，每一轮对话的经济成本。

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_experimental.utilities import PythonREPL
from langchain.agents import create_agent # 使用你环境中的 create_agent
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

model = init_chat_model(
    model="agnes-2.0-flash",
    model_provider="openai",
    base_url=os.getenv("AGNES_BASE_URL"),
    api_key=os.getenv("AGNES_API_KEY"),
)

repl = PythonREPL()

@tool
def python_analyst(code: str):
    """
    执行 Python 代码进行数据分析。你可以访问 'enterprise_data/' 目录下的文件。
    """
    return repl.run(code)

tools = [python_analyst]
print("环境初始化完成。")

环境初始化完成。


## 1. 构造 Mega-Prompt (全量规则注入)
这是 Alex 呕心沥血编写的“上帝指令”。请注意它包含了多少不同维度的规则。

In [2]:
MEGA_SYSTEM_PROMPT = """
你是一位 GlobalCorp 的全能分析大师。你必须严格遵守以下所有部门的规则：

### 1. 财务部规则 (Finance Rules)
- 必须计算 Net_Revenue。公式：(Qty * Price) - Discount。
- EMEA 地区税率 15%，US 地区 8%，APAC 地区 12%。
- 所有金额必须保留 2 位小数。

### 2. 市场部规则 (Marketing Rules)
- 将 'market' 映射为 'Region'，'qty' 映射为 'Quantity'。
- 如果产品是 'Software' 且金额 > 1000，标记为 'High Priority'。
- 必须清洗价格列中的 $ 符号和逗号。

### 3. CEO 视觉规范 (Executive Visual Rules)
- 报表输出必须包含一段中文总结。
- 必须打印出每个 Region 的平均单价 (Average Unit Price)。
- 如果某个 Region 收入低于 500，必须发出 'UNDER_PERFORMING' 警告。

### 4. 法务与合规 (Compliance Rules)
- 禁止在控制台打印 TransactionID (隐私保护)。
- 禁止修改原始 CSV 文件，必须在 DataFrame 副本上操作。

### 5. 更多细碎规则...
- (此处省略另外 20 条模拟规则以节省空间，但请想象它们的存在)
"""

## 2. 挑战 Agent：在混乱中执行任务
我们给 Agent 一个既涉及财务核算，又涉及 CEO 警告，还涉及市场清洗的任务。

In [3]:
# 使用 system_prompt 参数创建 Agent
agent = create_agent(
    model,
    tools,
    system_prompt=MEGA_SYSTEM_PROMPT  # 这里的参数名根据你的反馈设为 system_prompt
)

query = "请分析 modern_marketing.csv，计算净收入，并告诉我是否有任何 Region 表现不佳？"

print("--- Agent 开始处理庞大的指令集 ---\n")
result = agent.invoke({"messages": [("human", query)]})

print("\n--- 最终输出 ---\n")
print(result["messages"][-1].content)

--- Agent 开始处理庞大的指令集 ---



Python REPL can execute arbitrary code. Use with caution.



--- 最终输出 ---

### 分析报告

**1. 净收入计算 (Net Revenue)**
根据公式 `(Qty * Price) - Discount` 计算各地区的净收入如下：
*   **EMEA**: 2490.00
*   **LATAM**: 2050.00
*   **US**: 9900.00

**2. 各地区平均单价 (Average Unit Price)**
*   **EMEA**: 1250.00
*   **LATAM**: 2100.00
*   **US**: 500.00

**3. 表现不佳警告 (UNDER_PERFORMING)**
*   **结论**：没有任何 Region 的总净收入低于 500。
*   **状态**：所有地区均表现正常，未触发 `UNDER_PERFORMING` 警告。

**中文总结**：
本次分析显示，GlobalCorp 的三个主要市场区域（EMEA、LATAM、US）均实现了正向净收入。其中 US 地区贡献了最高的净收入（9900.00），而 LATAM 地区的平均单价最高（2100.00）。根据设定的标准（净收入低于 500 视为表现不佳），目前没有任何地区被标记为表现不佳。所有数据均已在 DataFrame 副本上处理，未修改原始文件，且未打印任何 TransactionID 以符合合规要求。


## 3. 实验观察：崩溃的迹象

请仔细观察 Agent 的输出。在规则极其密集的情况下，你可能会发现：
1. **遗漏规则**：它可能算对了税，但忘了打印“平均单价”，或者忘了检查“表现不佳”的警告。
2. **Token 爆炸**：每一次简单的对话，你都要支付 `MEGA_SYSTEM_PROMPT` 全文的费用。在企业级应用中，这会导致账单飞涨。
3. **维护噩梦**：如果财务部要修改一个税率，Alex 必须在这一大坨文本里寻找并修改。如果改错了，可能会导致市场部的逻辑也崩溃。

### 核心结论
**超级提示词 (Mega-Prompt) 无法支持复杂的生产系统。** 我们需要一种方式，让 Agent 能够：
- **平时只知道有哪些技能 (Metadata Only)**
- **用到时才读取具体规则 (Progressive Disclosure)**

**这就是 DeepAgent Skills 的用武之地。在最后的 Notebook 5 中，我们将把这一大坨指令拆解成一个个独立的 `SKILL.md`，实现真正的模块化智能。**